In [1]:
import ast
import json
from collections import defaultdict
from dotenv import load_dotenv
import os
from openai import OpenAI
from langgraph.graph import StateGraph, END
from langchain_core.runnables import RunnableLambda
from pydantic import BaseModel
from typing import List, Dict, Any

# --- Load API key ---
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

# --- LangGraph State & Models ---
class ArticleInput(BaseModel):
    articles: List[Dict[str, str]]

class SummaryState(BaseModel):
    input: ArticleInput
    output: str = None  # Raw response_text from LLM

class Section(BaseModel):
    subheading: str
    bullets: List[str]

class StructuredSummary(BaseModel):
    topic_title: str
    summary: str
    content: List[Section]

# --- Summarization Node ---
def generate_structured_summary(state: SummaryState) -> SummaryState:
    articles = state.input.articles
    combined_text = "\n\n".join([f"Title: {a['title']}\nContent: {a['content']}" for a in articles])

    prompt = f"""

    These are some articles related to a common topic. Your task is to extract important facts from all the articles and prepare a fact report. Do NOT give individual article summary but try to connect all of them and present in a structured format. include the facts given in the articles under these headings.
Remove emojees. Identify what part/subject of the articles is unrelated to the overall theme of the article, and don't include that in the final output. There is no length restriction on the report but keep it concise and judge the importance of the facts before writing in the report. Don't include the points which are not in the provided news articles. If you feel numbers are important for the news, then include the ballpark figures instead of writing all the figures.
Some examples of important facts can be important quotes spoken by leaders, and anything which can have a significant impact.
    It the incident is a tragedy/celebration(hard news) include the person being credited for that, as per the news articles and the reason/explanation that is happening. Include the sentences which may affect people's lives directly. Include the timeline which led to the event.
    For investigative news, include the legal and social details of the news and how are things being reformed. also identify the sources of information like how was this uncovered.
    For business news, if any market trend or impact on financial market is present, then include it. include the potential developments in business, stock markets, investments, mergers, and acquisitions which can happen in the future because of it.
    For local(district level/village level/small town level) news, if present, then include the facts related to local leadership, regional voter base, and public opinion.
    If present in any article, include how the public is reacting to the news, like protests, celebrations, or any other form of public sentiment.
    If news is about political party and elections, then identify the individual party strategies and include it in the report
The facts need to be divided into different headings ensuring the flow, consistency and coherence. The points under the headings should be relevant and accurate. Scan through all the articles to find the facts which can come under that heading.
The final output of the report should look like -


Output format:
{{
  "topic_title": "...",
  "summary": "...",
  "content": [
    {{
      "subheading": "Heading 1",
      "bullets": ["Fact 1", "Fact 2"]
    }},
    ...
  ]
}}

"topic_title" value will be the {{Title of the report generated based on the articles passed as input}}
"summary" value will be the {{Summary of the topic in 100 words conveying the gist of the articles}}
"subheading" value will be the {{Heading 1 - a heading covering the pointers }}
"bullets" value will be the {{Facts written as pointers under heading 1}}
There will be multiple subheadings and bullets in the content array
 ...

Articles:
{combined_text}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5,
    )

    state.output = response.choices[0].message.content.strip()
    return state

# --- LangGraph Setup ---
graph_builder = StateGraph(SummaryState)
graph_builder.add_node("summarizer", RunnableLambda(generate_structured_summary))
graph_builder.set_entry_point("summarizer")
graph_builder.add_edge("summarizer", END)
graph = graph_builder.compile()

# --- Load and Group Articles by Topic ---
with open("processed_articles_with_topics.json", "r", encoding="utf-8") as f:
    all_articles = json.load(f)

topic_clusters = defaultdict(list)
for url, data in all_articles.items():
    topic = data.get("topic", "Unknown")
    content = data.get("content", "").strip()
    if content:
        topic_clusters[topic].append({
            "title": data.get("title", ""),
            "content": content,
            "url": url
        })

# --- Run Summarization and Parse Output ---
i = 0
for topic, articles in topic_clusters.items():
    if i == 1:
        break
    i += 1
    print(f"\n--- Generating summary for topic: {topic} ---")

    filtered_articles = [a for a in articles if len(a.get("content", "")) > 100]
    if len(filtered_articles) < 1:
        print("⚠️ Skipped due to insufficient content.")
        continue

    try:
        article_input = ArticleInput(articles=filtered_articles)
        state = SummaryState(input=article_input)
        result = graph.invoke(state)

        raw_response = result["output"]  # ✅ correct
        try:
            cleaned = raw_response.strip()
            if cleaned.startswith("```python"):
                cleaned = cleaned[len("```python"):].strip()
            if cleaned.endswith("```"):
                cleaned = cleaned[:-3].strip()
            parsed_dict = ast.literal_eval(cleaned)
            structured = StructuredSummary(**parsed_dict)
            print("\n✅ Parsed Structured Summary:")
            print(json.dumps(structured.model_dump(), indent=2))  # use .dict() if pydantic v1
        except Exception as parse_err:
            print("⚠️ Failed to parse model output. Raw output below:")
            print(raw_response)
            print(f"Parsing error: {parse_err}")

    except Exception as e:
        print(f"❌ Error generating summary for topic '{topic}': {e}")
        


--- Generating summary for topic: Maharashtra Language ---

✅ Parsed Structured Summary:
{
  "topic_title": "Controversy Over Maharashtra's Three-Language Policy",
  "summary": "The Maharashtra government's decision to introduce Hindi as a third language in schools has sparked significant controversy, drawing criticism from various political leaders, educational bodies, and the public. The policy, which mandates Hindi for students in Classes 1 to 5, has faced backlash for being perceived as an imposition on the Marathi language. Amidst protests and calls for reconsideration, Chief Minister Devendra Fadnavis has announced that a final decision will be made after consulting stakeholders, including educators and literary figures. The debate highlights tensions surrounding language policy in a linguistically diverse state.",
  "content": [
    {
      "subheading": "Government's Three-Language Policy",
      "bullets": [
        "The Maharashtra government announced Hindi will be taught a

In [2]:
structured

StructuredSummary(topic_title="Controversy Over Maharashtra's Three-Language Policy", summary="The Maharashtra government's decision to introduce Hindi as a third language in schools has sparked significant controversy, drawing criticism from various political leaders, educational bodies, and the public. The policy, which mandates Hindi for students in Classes 1 to 5, has faced backlash for being perceived as an imposition on the Marathi language. Amidst protests and calls for reconsideration, Chief Minister Devendra Fadnavis has announced that a final decision will be made after consulting stakeholders, including educators and literary figures. The debate highlights tensions surrounding language policy in a linguistically diverse state.", content=[Section(subheading="Government's Three-Language Policy", bullets=['The Maharashtra government announced Hindi will be taught as a third language in Marathi and English medium schools for Classes 1 to 5.', 'The policy requires consent from at